
   # 字符级 MLP 语言模型 · 回顾笔记

   **任务**：用前 3 个字符，预测下一个字符（`block_size=3, emb_size=10, hidden=200`）

   ## 5 个参数

   | 张量 | 形状 | 角色 |
   |---|---|---|
   | `C` | 27 × emb | 字符嵌入表，每字符查一行得 emb 维向量 |
   | `W1` | emb·bs × hidden | 输入 → 特征，**每列 = 一种特征的模板** |
   | `b1` | hidden | 每个特征的触发门槛 |
   | `W2` | hidden × 27 | 特征 → 字符得分，**每列 = 一个字符的打分规则** |
   | `b2` | 27 | 每个字符的基线偏好 |

   ## 数据流

   ```
   3 个字符索引
    → 查表 C        3×emb（各查各的，隔离）
    → 拼接          emb·bs 维
    → @W1 + b1      hidden 维
    → tanh          压到 −1~1（特征激活）
    → @W2 + b2      27 维 logits（原始得分，无界）
    → softmax       27 个概率
    → 取最大/采样    下一个字符
   ```

   ## 5 条关键直觉

   1. **矩阵 = 一束线**。`W1[r][h]` = 嵌入神经元 r → 隐藏神经元 h 的一条线；行=左端、列=右端、格子=一条线。`h[h] = Σ_r emb[r]·W1[r][h]` 就是这个神经元
 所有入边的加权和。
   2. **W1 每列 = 一种特征**。列向量是"要找的模板"，激活值是"匹配度"（点积=相似度）。语义靠训练学出来，初始化时只是随机数。
   3. **字符不隔离**。查表时各查各的，但拼接后被 W1 一起算。证据：W1 输入维度是 `emb_size * block_size`——**乘法即拼接**。把 W1 每列按 bs 切段，每段 =
 该特征对某个位置的权重。
   4. **W1 看什么，W2 怎么下结论**。W1 提取特征，W2 做读出：`W2[特征][字符]` = 这个特征给该字符投几票，`b2` = 起跑线。输出层保持线性，才让 logit 拉得开
 差距；非线性交给 softmax。
   5. **logits ≠ log**。logits 是 softmax 前的无界原始得分；词源是 logistic 回归的 log-odds，本质是"未归一化的对数概率"。

   ## 训练

   - 全部 5 组进入 `parameters`，用**同一个 loss 联合训练**（`param ← param − lr·grad`），所以 W1 的特征和 W2 的规则是配套演化出来的。
   - **梯度流决定学习难度**：W2/b2 离 loss 最近 → 学得快；W1/b1 要穿 tanh 和 W2；`C` 在最深处 → 学得慢（常给更大 lr 或预训练）。
   - `hidden` 100→500：`W1`(emb·bs·hidden) 和 `W2`(hidden·27) 参数量**各翻 5 倍**，参数主要吃在这两层。

   ## 一句话总纲

   > **3 个字符拼接 → W1 提取"哪种特征出现了" → tanh 给有界激活 → W2 让每个特征给 27 个候选字符投票 → softmax 定胜负 → 猜下一个字。全部参数联合训练。**